#Dữ liệu lịch sử giá

In [ ]:
import requests
import pandas as pd
import numpy as np
import time
from tqdm import tqdm

In [ ]:
# ─── CẤU HÌNH ────────────────────────────────────────────────────────
SYMBOL   = "BTCUSDT"
INTERVAL = "30m"
LIMIT    = 1000  # Số nến tối đa mỗi lần gọi API (Binance cho phép tối đa 1500)
URL      = "https://api.binance.com/api/v3/klines"

# Khớp đúng khoảng thời gian tweet của bạn (có buffer 1 ngày 2 đầu để chắc chắn)
START = pd.Timestamp("2021-02-04", tz="UTC")
END   = pd.Timestamp("2023-01-10", tz="UTC")

start_ms     = int(START.timestamp() * 1000)
end_ms       = int(END.timestamp()   * 1000)
ms_per_candle = 30 * 60 * 1000  # 30 phút tính bằng milliseconds

# Ước tính số lượng request
est_candles  = (end_ms - start_ms) // ms_per_candle
est_requests = (est_candles // LIMIT) + 1

print(f"📅 Phạm vi tải   : {START.date()} → {END.date()}")
print(f"🕯️  Nến 30m ước tính: {est_candles:,} nến")
print(f"🌐 Số lần gọi API: ~{est_requests} lần")

# ─── HÀM GỌI API CÓ TỰ ĐỘNG RETRY ───────────────────────────────────
def fetch_klines(start_ms, end_ms, retries=3):
    """Gọi Binance API với cơ chế tự động thử lại khi gặp lỗi mạng."""
    params = {
        "symbol"   : SYMBOL,
        "interval" : INTERVAL,
        "startTime": start_ms,
        "endTime"  : end_ms,
        "limit"    : LIMIT
    }
    for attempt in range(retries):
        try:
            resp = requests.get(URL, params=params, timeout=10)
            resp.raise_for_status()
            return resp.json()
        except requests.RequestException as e:
            if attempt < retries - 1:
                wait = 2 ** attempt  # Exponential backoff: 1s, 2s, 4s
                print(f"\n  ⚠️  Lỗi: {e}. Thử lại sau {wait}s...")
                time.sleep(wait)
            else:
                raise

# ─── TẢI DỮ LIỆU VỚI PHÂN TRANG TỰ ĐỘNG ─────────────────────────────
print(f"\nĐang tải dữ liệu giá BTC từ Binance...")
all_klines    = []
current_start = start_ms

with tqdm(total=est_requests, desc="Gọi Binance API", unit="req") as pbar:
    while current_start < end_ms:
        data = fetch_klines(current_start, end_ms)

        if not data:
            break

        all_klines.extend(data)

        # Dịch chuyển con trỏ đến nến TIẾP THEO sau batch vừa nhận
        current_start = data[-1][0] + ms_per_candle

        pbar.update(1)
        pbar.set_postfix({"nến": f"{len(all_klines):,}"})

        # Ngủ 0.12s để tôn trọng rate limit của Binance (tối đa ~1200 req/phút)
        time.sleep(0.12)

# ─── XÂY DỰNG DATAFRAME ──────────────────────────────────────────────
print(f"\nĐã tải xong {len(all_klines):,} nến thô. Đang xây dựng DataFrame...")

COLUMNS = [
    'open_time', 'open', 'high', 'low', 'close', 'volume',
    'close_time', 'quote_volume', 'trades',
    'taker_buy_base', 'taker_buy_quote', 'ignore'
]

df_btc = pd.DataFrame(all_klines, columns=COLUMNS)

# Chuyển đổi kiểu dữ liệu
df_btc['open_time']  = pd.to_datetime(df_btc['open_time'],  unit='ms', utc=True)
df_btc['close_time'] = pd.to_datetime(df_btc['close_time'], unit='ms', utc=True)

for col in ['open', 'high', 'low', 'close', 'volume', 'quote_volume']:
    df_btc[col] = df_btc[col].astype(float)

df_btc['trades'] = df_btc['trades'].astype(int)

# Loại bỏ cột không dùng & trùng lặp
df_btc = (df_btc
    .drop(columns=['ignore', 'taker_buy_base', 'taker_buy_quote'])
    .drop_duplicates(subset='open_time')
    .sort_values('open_time')
    .reset_index(drop=True)
)

📅 Phạm vi tải   : 2021-02-04 → 2023-01-10
🕯️  Nến 30m ước tính: 33,840 nến
🌐 Số lần gọi API: ~34 lần

Đang tải dữ liệu giá BTC từ Binance...


Gọi Binance API: 100%|██████████| 34/34 [00:08<00:00,  3.88req/s, nến=33,810]



Đã tải xong 33,810 nến thô. Đang xây dựng DataFrame...


In [ ]:
# ─── LƯU LÊN GOOGLE DRIVE ────────────────────────────────────────────
df_btc.to_csv('/content/drive/MyDrive/Combine_dataset/historical_data.csv', index=False)
print(f"✅ Đã lưu tại: {SAVE_PATH}")

# ─── BÁO CÁO KẾT QUẢ ─────────────────────────────────────────────────
print("\n" + "═"*60)
print("  BÁO CÁO DỮ LIỆU GIÁ BTC-USDT 30M")
print("═"*60)
print(f"  Tổng số nến tải được : {len(df_btc):,}")
print(f"  Nến đầu tiên         : {df_btc['open_time'].min()}")
print(f"  Nến cuối cùng        : {df_btc['open_time'].max()}")
print(f"  Giá thấp nhất (low)  : ${df_btc['low'].min():,.2f}")
print(f"  Giá cao nhất (high)  : ${df_btc['high'].max():,.2f}")
print("═"*60)

df_btc[['open_time', 'open', 'high', 'low', 'close', 'volume', 'trades']].head()

✅ Đã lưu tại: /content/drive/MyDrive/Combine_dataset/historical_data

════════════════════════════════════════════════════════════
  BÁO CÁO DỮ LIỆU GIÁ BTC-USDT 30M
════════════════════════════════════════════════════════════
  Tổng số nến tải được : 33,810
  Nến đầu tiên         : 2021-02-04 00:00:00+00:00
  Nến cuối cùng        : 2023-01-10 00:00:00+00:00
  Giá thấp nhất (low)  : $15,476.00
  Giá cao nhất (high)  : $69,000.00
════════════════════════════════════════════════════════════


,open_time,open,high,low,close,volume,trades
0,2021-02-04 00:00:00+00:00,37620.26,37994.47,37615.93,37989.68,2505.803253,56371
1,2021-02-04 00:30:00+00:00,37989.67,38128.00,37800.00,38063.15,2455.389346,61573
2,2021-02-04 01:00:00+00:00,38065.00,38272.36,37861.83,38069.74,2393.684263,53217
3,2021-02-04 01:30:00+00:00,38069.75,38288.00,37878.27,38147.35,1901.193393,39076
4,2021-02-04 02:00:00+00:00,38147.36,38215.40,37505.51,37708.97,2597.024977,53925


#Dữ liệu cảm xúc

In [ ]:
df_sentiment = pd.read_csv('/content/drive/MyDrive/Combine_dataset/historical_data.csv')

Gom cụm dữ liệu cảm xúc theo khoảng thời gian 30 phút, khoảng thời gian nào không có tweet nào được đăng thì bỏ

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# BƯỚC 9 — GOM CỤM CẢM XÚC THEO KHUNG THỜI GIAN 30 PHÚT
# • Mean Weighting    : Σs(xi) / n
# • Influence Weighting: Σs(xi)·(ln(wi+1)+1) / Σ(ln(wi+1)+1)
# Các khung không có tweet → tự động bị loại bỏ bởi groupby
# ══════════════════════════════════════════════════════════════════════

# ─── 1. CHUẨN BỊ CỘT THỜI GIAN ──────────────────────────────────────
df_sentiment['date']    = pd.to_datetime(df_sentiment['date'], utc=True)
df_sentiment['date_30m'] = df_sentiment['date'].dt.floor('30min')  # Làm tròn xuống mốc 30 phút gần nhất
df_sentiment['weight']  = df_sentiment['weight'].fillna(1.0)        # Đảm bảo không có NaN ở denominator

# ─── 2. ĐỊNH NGHĨA CÁC CỘT SENTIMENT & TÊN ĐẦU RA ───────────────────
# Mỗi cột sentiment sẽ cho ra 2 cột đầu ra:
#   <short>_mean       ← Mean Weighting (cách 1)
#   <short>_influence  ← User Influence Weighting
SENTIMENT_MAP = {
    'vader_compound' : 'vader',
    'bert_compound'  : 'bert',
    'ed_joy'         : 'ed_joy',
    'ed_sadness'     : 'ed_sadness',
    'ed_anger'       : 'ed_anger',
    'ed_surprise'    : 'ed_surprise',
    'ed_disgust'     : 'ed_disgust',
    'ed_fear'        : 'ed_fear',
    'ed_others'      : 'ed_others',
}

# ─── 3. TÍNH TRƯỚC TỬ SỐ INFLUENCE: s(xi) × weight ──────────────────
# Thực hiện trước để tận dụng vectorized operation của Pandas (nhanh nhất)
for col in SENTIMENT_MAP:
    df_sentiment[f'_ws_{col}'] = df_sentiment[col] * df_sentiment['weight']

# ─── 4. XÂY DỰNG AGGREGATION SPECS VÀ GROUPBY ────────────────────────
agg_specs = {
    'tweet_count' : ('vader_compound', 'count'),  # Số tweet trong khung 30m
    'weight_sum'  : ('weight', 'sum'),             # Σweight (mẫu số Influence)
}
for col, short in SENTIMENT_MAP.items():
    agg_specs[f'{short}_mean']   = (col,          'mean')  # Cách 1: Mean
    agg_specs[f'_wsum_{short}']  = (f'_ws_{col}', 'sum')   # Σ(s × w) cho Influence

# groupby chỉ tạo row cho những khung 30m THỰC SỰ có tweet
# → Các khoảng không có tweet bị loại bỏ tự động
df_agg = df_sentiment.groupby('date_30m').agg(**agg_specs).reset_index()

# ─── 5. TÍNH USER INFLUENCE WEIGHTING ────────────────────────────────
# Công thức: Σ(s·w) / Σw
for col, short in SENTIMENT_MAP.items():
    df_agg[f'{short}_influence'] = df_agg[f'_wsum_{short}'] / df_agg['weight_sum']

# Dọn dẹp các cột tạm thời
temp_cols = [f'_wsum_{s}' for s in SENTIMENT_MAP.values()] + ['weight_sum']
df_agg = df_agg.drop(columns=temp_cols)

# Dọn dẹp các cột trung gian trong df gốc
df_sentiment = df_sentiment.drop(columns=[f'_ws_{col}' for col in SENTIMENT_MAP])

# ─── 6. BÁO CÁO KẾT QUẢ ─────────────────────────────────────────────
total_possible = int((df_agg['date_30m'].max() - df_agg['date_30m'].min())
                     .total_seconds() / 1800) + 1

print("═"*65)
print("  BÁO CÁO GOM CỤM CẢM XÚC THEO KHUNG 30 PHÚT — BƯỚC 9")
print("═"*65)
print(f"  Từ                         : {df_agg['date_30m'].min()}")
print(f"  Đến                        : {df_agg['date_30m'].max()}")
print(f"  Tổng khung 30m lý thuyết   : {total_possible:,}")
print(f"  Khung 30m có tweet (giữ lại): {len(df_agg):,}")
print(f"  Khung 30m không có tweet   : {total_possible - len(df_agg):,}  ← đã loại bỏ")
print(f"  Số tweet trung bình/khung  : {df_agg['tweet_count'].mean():.1f}")
print(f"  Số tweet nhiều nhất/khung  : {df_agg['tweet_count'].max():,}")
print("═"*65)
print(f"\n  Tổng số cột: {len(df_agg.columns)}")
print(f"  Danh sách : {list(df_agg.columns)}")

═════════════════════════════════════════════════════════════════
  BÁO CÁO GOM CỤM CẢM XÚC THEO KHUNG 30 PHÚT — BƯỚC 9
═════════════════════════════════════════════════════════════════
  Từ                         : 2021-02-05 10:30:00+00:00
  Đến                        : 2023-01-09 23:30:00+00:00
  Tổng khung 30m lý thuyết   : 33,771
  Khung 30m có tweet (giữ lại): 8,024
  Khung 30m không có tweet   : 25,747  ← đã loại bỏ
  Số tweet trung bình/khung  : 399.5
  Số tweet nhiều nhất/khung  : 5,856
═════════════════════════════════════════════════════════════════

  Tổng số cột: 20
  Danh sách : ['date_30m', 'tweet_count', 'vader_mean', 'bert_mean', 'ed_joy_mean', 'ed_sadness_mean', 'ed_anger_mean', 'ed_surprise_mean', 'ed_disgust_mean', 'ed_fear_mean', 'ed_others_mean', 'vader_influence', 'bert_influence', 'ed_joy_influence', 'ed_sadness_influence', 'ed_anger_influence', 'ed_surprise_influence', 'ed_disgust_influence', 'ed_fear_influence', 'ed_others_influence']


In [ ]:
df_agg.head()

,date_30m,tweet_count,vader_mean,bert_mean,ed_joy_mean,ed_sadness_mean,ed_anger_mean,ed_surprise_mean,ed_disgust_mean,ed_fear_mean,ed_others_mean,vader_influence,bert_influence,ed_joy_influence,ed_sadness_influence,ed_anger_influence,ed_surprise_influence,ed_disgust_influence,ed_fear_influence,ed_others_influence
0,2021-02-05 10:30:00+00:00,7,0.499743,0.496921,0.218451,0.001815,0.002062,0.004059,0.002023,0.070462,0.701128,0.523925,0.476479,0.200395,0.002106,0.002193,0.004380,0.002182,0.110912,0.677832
1,2021-02-05 11:00:00+00:00,38,0.136595,0.185188,0.051286,0.001500,0.001826,0.003833,0.006846,0.003583,0.931126,0.126673,0.177932,0.049518,0.001483,0.001801,0.003901,0.006981,0.003479,0.932837
2,2021-02-05 11:30:00+00:00,32,0.073659,0.206985,0.067796,0.001879,0.002677,0.034898,0.026868,0.033238,0.832645,0.107015,0.233185,0.063925,0.001755,0.002249,0.036392,0.015516,0.030965,0.849197
3,2021-02-05 12:00:00+00:00,56,0.105086,0.173712,0.045141,0.001548,0.002745,0.022557,0.040563,0.002911,0.884536,0.113878,0.165533,0.052328,0.001531,0.002608,0.020162,0.038244,0.002826,0.882302
4,2021-02-05 12:30:00+00:00,60,0.313377,0.412711,0.166651,0.001419,0.018374,0.004539,0.032040,0.005374,0.771604,0.296935,0.408650,0.167453,0.001417,0.020831,0.004614,0.035015,0.005969,0.764700


In [ ]:
# Lưu lại
SAVE_PATH = "/content/drive/MyDrive/Combine_dataset/sentiment_aggregated_30m.csv"
df_agg.to_csv(SAVE_PATH, index=False)
print(f"\n✅ Đã lưu tại: {SAVE_PATH}")


✅ Đã lưu tại: /content/drive/MyDrive/Combine_dataset/sentiment_aggregated_30m.csv


#Kết hợp 2 dataset lịch sử giá và cảm xúc

In [ ]:
df_agg = pd.read_csv("/content/drive/MyDrive/Combine_dataset/sentiment_aggregated_30m.csv")
df_btc = pd.read_csv("/content/drive/MyDrive/Combine_dataset/historical_data.csv")

In [ ]:
df_agg.shape

(8024, 20)

In [ ]:
df_btc.shape

(33810, 9)

In [ ]:
# 1. Đưa Index quay trở lại làm cột bình thường nếu mốc thời gian đang nằm ở Index
df_agg_temp = df_agg.reset_index()
df_btc_temp = df_btc.reset_index()

# 2. In ra danh sách cột thực tế để chúng ta đối chiếu
print("="*65)
print("  TÊN CÁC CỘT THỰC TẾ TRONG BẢNG GIÁ (df_btc):")
print(list(df_btc.columns))
print("\n  TÊN CÁC CỘT THỰC TẾ TRONG BẢNG CẢM XÚC (df_agg):")
print(list(df_agg_temp.columns))
print("="*65)

# 3. Thử tìm cột mốc thời gian tự động
# Chúng ta sẽ tìm cột chứa các từ khóa liên quan đến thời gian như 'date', 'time', 'timestamp'
btc_time_col = [col for col in df_btc_temp.columns if any(x in col.lower() for x in ['time', 'date', 'timestamp'])][0]
agg_time_col = [col for col in df_agg_temp.columns if any(x in col.lower() for x in ['time', 'date', 'timestamp'])][0]

price_min = df_btc_temp[btc_time_col].min()
price_max = df_btc_temp[btc_time_col].max()
tweet_min = df_agg_temp[agg_time_col].min()
tweet_max = df_agg_temp[agg_time_col].max()

print("\n    BÁO CÁO ĐỐI CHIẾU DẢI THỜI GIAN (TỰ ĐỘNG KHỚP CỘT)")
print("="*65)
print(f"  Cột thời gian df_btc  : '{btc_time_col}'")
print(f"  Cột thời gian df_agg  : '{agg_time_col}'")
print(f"  Dữ liệu Giá (Binance) : Từ {price_min} đến {price_max}")
print(f"  Dữ liệu Cảm xúc (Agg) : Từ {tweet_min} đến {tweet_max}")
print("="*65)

  TÊN CÁC CỘT THỰC TẾ TRONG BẢNG GIÁ (df_btc):
['open_time', 'open', 'high', 'low', 'close', 'volume', 'close_time', 'quote_volume', 'trades']

  TÊN CÁC CỘT THỰC TẾ TRONG BẢNG CẢM XÚC (df_agg):
['index', 'date_30m', 'tweet_count', 'vader_mean', 'bert_mean', 'ed_joy_mean', 'ed_sadness_mean', 'ed_anger_mean', 'ed_surprise_mean', 'ed_disgust_mean', 'ed_fear_mean', 'ed_others_mean', 'vader_influence', 'bert_influence', 'ed_joy_influence', 'ed_sadness_influence', 'ed_anger_influence', 'ed_surprise_influence', 'ed_disgust_influence', 'ed_fear_influence', 'ed_others_influence']

    BÁO CÁO ĐỐI CHIẾU DẢI THỜI GIAN (TỰ ĐỘNG KHỚP CỘT)
  Cột thời gian df_btc  : 'open_time'
  Cột thời gian df_agg  : 'date_30m'
  Dữ liệu Giá (Binance) : Từ 2021-02-04 00:00:00+00:00 đến 2023-01-10 00:00:00+00:00
  Dữ liệu Cảm xúc (Agg) : Từ 2021-02-05 10:30:00+00:00 đến 2023-01-09 23:30:00+00:00


Kết hợp

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# BƯỚC 10 — KẾT HỢP DỮ LIỆU GIÁ BTC VỚI SENTIMENT
# Theo Section 3.3.4: merge by matching timestamps → inner join
# Chỉ giữ những khung 30m thực sự có tweet (đúng theo bài báo)
# ══════════════════════════════════════════════════════════════════════

# ─── 1. ĐẢM BẢO DATETIME UTC KHỚP NHAU ──────────────────────────────
df_btc['open_time'] = pd.to_datetime(df_btc['open_time'], utc=True)
df_agg['date_30m']  = pd.to_datetime(df_agg['date_30m'],  utc=True)

# ─── 2. INNER JOIN — Chỉ giữ mốc 30m có CẢ giá BTC lẫn sentiment ────
df_final = pd.merge(
    df_btc[['open_time', 'open', 'high', 'low', 'close', 'volume']],
    df_agg,
    left_on  = 'open_time',
    right_on = 'date_30m',
    how      = 'inner'
).drop(columns=['date_30m']).rename(columns={'open_time': 'date_30m'})

# Sắp xếp theo thời gian tăng dần
df_final = df_final.sort_values('date_30m').reset_index(drop=True)

# ─── 3. BÁO CÁO ──────────────────────────────────────────────────────
print("═"*60)
print("  BÁO CÁO DATASET CUỐI CÙNG — SAU KHI MERGE")
print("═"*60)
print(f"  df_btc  (giá BTC 30m)     : {len(df_btc):>7,} dòng")
print(f"  df_agg  (sentiment 30m)   : {len(df_agg):>7,} dòng")
print(f"  df_final (sau inner join) : {len(df_final):>7,} dòng  ✅")
print(f"\n  Từ : {df_final['date_30m'].min()}")
print(f"  Đến: {df_final['date_30m'].max()}")
print(f"\n  Số cột: {len(df_final.columns)}")
print(f"  Cột  : {list(df_final.columns)}")
print("═"*60)

# ─── 4. KIỂM TRA NHANH ───────────────────────────────────────────────
print(f"\nGiá BTC (close): min=${df_final['close'].min():,.0f}  |  max=${df_final['close'].max():,.0f}")
print(f"Tweet/khung 30m: min={df_final['tweet_count'].min()}  |  mean={df_final['tweet_count'].mean():.1f}  |  max={df_final['tweet_count'].max():,}")

════════════════════════════════════════════════════════════
  BÁO CÁO DATASET CUỐI CÙNG — SAU KHI MERGE
════════════════════════════════════════════════════════════
  df_btc  (giá BTC 30m)     :  33,810 dòng
  df_agg  (sentiment 30m)   :   8,024 dòng
  df_final (sau inner join) :   8,019 dòng  ✅

  Từ : 2021-02-05 10:30:00+00:00
  Đến: 2023-01-09 23:30:00+00:00

  Số cột: 25
  Cột  : ['date_30m', 'open', 'high', 'low', 'close', 'volume', 'tweet_count', 'vader_mean', 'bert_mean', 'ed_joy_mean', 'ed_sadness_mean', 'ed_anger_mean', 'ed_surprise_mean', 'ed_disgust_mean', 'ed_fear_mean', 'ed_others_mean', 'vader_influence', 'bert_influence', 'ed_joy_influence', 'ed_sadness_influence', 'ed_anger_influence', 'ed_surprise_influence', 'ed_disgust_influence', 'ed_fear_influence', 'ed_others_influence']
════════════════════════════════════════════════════════════

Giá BTC (close): min=$15,650  |  max=$66,876
Tweet/khung 30m: min=1  |  mean=399.7  |  max=5,856


In [ ]:
# ─── 5. LƯU LÊN GOOGLE DRIVE ─────────────────────────────────────────
SAVE_PATH = "/content/drive/MyDrive/Combine_dataset/btc_final_dataset.csv"
df_final.to_csv(SAVE_PATH, index=False)
print(f"\n✅ Đã lưu tại: {SAVE_PATH}")


✅ Đã lưu tại: /content/drive/MyDrive/Combine_dataset/btc_final_dataset.csv


In [ ]:
df_final.tail()

,date_30m,open,high,low,close,volume,tweet_count,vader_mean,bert_mean,ed_joy_mean,...,ed_others_mean,vader_influence,bert_influence,ed_joy_influence,ed_sadness_influence,ed_anger_influence,ed_surprise_influence,ed_disgust_influence,ed_fear_influence,ed_others_influence
8014,2023-01-09 21:30:00+00:00,17232.66,17235.85,17165.04,17182.48,5524.35573,486,0.192054,0.214941,0.113077,...,0.812102,0.180829,0.199403,0.096234,0.002344,0.012289,0.004908,0.045847,0.009110,0.829268
8015,2023-01-09 22:00:00+00:00,17182.36,17211.01,17167.35,17209.11,3296.05357,466,0.144275,0.199785,0.096241,...,0.808853,0.146505,0.198546,0.091138,0.001748,0.011265,0.018378,0.044840,0.021056,0.811576
8016,2023-01-09 22:30:00+00:00,17209.11,17222.32,17190.27,17204.83,2148.19224,390,0.165060,0.194575,0.069723,...,0.842517,0.164653,0.198987,0.068246,0.004657,0.011402,0.006484,0.042550,0.018008,0.848654
8017,2023-01-09 23:00:00+00:00,17204.83,17205.57,17128.00,17171.08,4777.83246,401,0.180116,0.217586,0.085975,...,0.835375,0.182591,0.218306,0.088281,0.007922,0.013319,0.005499,0.041017,0.008532,0.835432
8018,2023-01-09 23:30:00+00:00,17171.08,17198.14,17169.90,17178.26,2715.04756,411,0.143539,0.160108,0.073525,...,0.805883,0.135128,0.163211,0.073475,0.006213,0.009746,0.007568,0.073551,0.014397,0.815050


In [ ]:
df_final.shape

(8019, 25)